In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline
import numpy as np
from sklearn.metrics import f1_score
import pandas as pd
import torch

C:\Users\82104\anaconda3\envs\hanyang_paper\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# 1. 모델 불러오기
model_id = "searle-j/kote_for_easygoing_people"
tok = AutoTokenizer.from_pretrained(model_id)
mdl = AutoModelForSequenceClassification.from_pretrained(model_id)

pipe = TextClassificationPipeline(
    model=mdl,
    tokenizer=tok,
    return_all_scores=True,
    function_to_apply="sigmoid"
)

Device set to use cuda:0
C:\Users\82104\anaconda3\envs\hanyang_paper\Lib\site-packages\transformers\pipelines\text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [6]:
# 2. 데이터 불러기기
from datasets import load_dataset

data_files = {
    "train": r"C:\Users\82104\anaconda_projects_files(2025)\hanyang_paper(RE)\KOTE\train.tsv",
    "validation": r"C:\Users\82104\anaconda_projects_files(2025)\hanyang_paper(RE)\KOTE\val.tsv",
    "test": r"C:\Users\82104\anaconda_projects_files(2025)\hanyang_paper(RE)\KOTE\test.tsv",
}

dataset = load_dataset("csv", data_files=data_files, delimiter="\t", header=None)
dataset = dataset.rename_column("0", "id")
dataset = dataset.rename_column("1", "text")
dataset = dataset.rename_column("2", "labels")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'labels'],
        num_rows: 40000
    })
    validation: Dataset({
        features: ['id', 'text', 'labels'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'text', 'labels'],
        num_rows: 5000
    })
})


In [13]:
# 1. gold 라벨 준비 (test split)
y_true = np.array(dataset["test"]["labels"])

In [15]:
import numpy as np

NUM_CLASSES = 44

def encode_labels(label_str):
    vec = np.zeros(NUM_CLASSES, dtype=int)
    if isinstance(label_str, str):
        for idx in label_str.split(","):
            if idx.strip().isdigit():
                vec[int(idx)] = 1
    return vec

# test split 라벨을 멀티핫으로 변환
y_true = np.array([encode_labels(x) for x in dataset["test"]["labels"]])

print("y_true shape:", y_true.shape)  # (5000, 44)


y_true shape: (5000, 44)


In [17]:
# train split에서 앞 20줄만 확인
for i in range(20):
    print("ID:", dataset["train"]["id"][i])
    print("TEXT:", dataset["train"]["text"][i])
    print("LABELS (원본):", dataset["train"]["labels"][i])
    print("---")


ID: 39087
TEXT: 내가 톰행크스를 좋아하긴 했나보다... 초기 영화 빼고는 다 봤네.
LABELS (원본): 2,13,15,16,29,39
---
ID: 30893
TEXT: 정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요 ㅠ 생각보다 스트레스 심한 직업군이라는 생각입니다.
LABELS (원본): 0,5,7,10,19,22,29,35,36,38
---
ID: 45278
TEXT: 새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수함일지도 모르겠습니다. 그걸 가능케 한 사랑에 경의를 표합니다.
LABELS (원본): 1,2,7
---
ID: 16398
TEXT: 미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ
LABELS (원본): 9,15,20,23,26,28,29
---
ID: 13653
TEXT: 네 맞습니다 플스는 역시 30프레임이 어울리죠 ㅎ
LABELS (원본): 1,2,8,9,11,13,15,16,28,29,32,40,42
---
ID: 13748
TEXT: 어릴 때 했던 건데 아직도 볼 때마다 뒤통수 얼얼함 ㅋㅋㅋㅋ
LABELS (원본): 2,15,23,24,25,28,33
---
ID: 23965
TEXT: 물갈이약 구매했어요...미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 그 동안 만족했었는데... 약간 불량들이 있네요... 1) 함께 구매한 스포이드에 구멍이 있네요... 2) 부화통에 같이오는 흡착고무가 불량이라 안붙어요... 3) 예전에 구매한 온도계의 온도가 안맞아요...
LABELS (원본): 0,10,33
---
ID: 45552
TEXT: 십일조는 겨우60인데? 나머지 다 어디감? 혹시 엄마아빠 그랜져따로타고, 외식자주하는거아님? 이런저런건 다빼고 십일조얘기만하는건 그냥 머리나쁜거ㅜ
LABELS (원본): 0,3,9,10,12,20,22,23
---
ID: 30219
TEXT: 90년 줘야 하는거 아닌가? 

# ppt에 쓸꺼(모델로 데이터셋 예측)

In [13]:
import torch
import numpy as np
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 모델 로드
model_id = "searle-j/kote_for_easygoing_people"
tok = AutoTokenizer.from_pretrained(model_id)
mdl = AutoModelForSequenceClassification.from_pretrained(model_id)
mdl.eval()

# 2. 디바이스 설정 (GPU 있으면 cuda, 없으면 cpu)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mdl.to(device)

# 3. 라벨 멀티핫 변환
NUM_CLASSES = 44
def encode_labels(label_str):
    vec = np.zeros(NUM_CLASSES, dtype=int)
    if isinstance(label_str, str):
        for idx in label_str.split(","):
            if idx.strip().isdigit():
                vec[int(idx)] = 1
    return vec

y_true = np.array([encode_labels(x) for x in dataset["test"]["labels"]])
texts = dataset["test"]["text"]

# 4. 배치 추론
batch_size = 32   # CPU면 8~16, GPU면 32~128 가능
y_pred = []

for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]
    enc = tok(batch_texts, return_tensors="pt", padding=True,
              truncation=True, max_length=512).to(device)
    with torch.no_grad():
        logits = mdl(**enc).logits
        probs = torch.sigmoid(logits).cpu().numpy()
    binary = (probs >= 0.4).astype(int)
    y_pred.append(binary)

y_pred = np.vstack(y_pred)

# 5. 성능 평가
macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
samples = f1_score(y_true, y_pred, average="samples", zero_division=0)

print(f"✅ Macro-F1: {macro:.4f}")
print(f"✅ Micro-F1: {micro:.4f}")
print(f"✅ Samples-F1: {samples:.4f}")


✅ Macro-F1: 0.5344
✅ Micro-F1: 0.6483
✅ Samples-F1: 0.6379


In [1]:
# 200개 먼저 돌려보기  
# texts = df["감상평"].fillna("").astype(str).tolist()[:200]

In [11]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# 1. 데이터 로드
df = pd.read_csv("movie_long_micro_panel.csv")

# 2. 감상평이 비어있지 않은 행만 필터링 후 300개 샘플링
df_nonnull = df[df["감상평"].notna()].copy()
df_sample = df_nonnull.sample(300, random_state=42).copy()

# 3. 모델 로드
model_id = "searle-j/kote_for_easygoing_people"
tok = AutoTokenizer.from_pretrained(model_id)
mdl = AutoModelForSequenceClassification.from_pretrained(model_id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mdl.to(device)
mdl.eval()
print(f"✅ Running on {device}")

# 4. 감정 라벨 목록
emotion_labels = [
 '불평/불만','환영/호의','감동/감탄','지긋지긋','고마움','슬픔','화남/분노','존경','기대감','우쭐댐/무시함',
 '안타까움/실망','비장함','의심/불신','뿌듯함','편안/쾌적','신기함/관심','아껴주는','부끄러움','공포/무서움','절망',
 '한심함','역겨움/징그러움','짜증','어이없음','없음','패배/자기혐오','귀찮음','힘듦/지침','즐거움/신남','깨달음',
 '죄책감','증오/혐오','흐뭇함(귀여움/예쁨)','당황/난처','경악','부담/안_내킴','서러움','재미없음','불쌍함/연민','놀람',
 '행복','불안/걱정','기쁨','안심/신뢰'
]

# 5. 텍스트 리스트
texts = df_sample["감상평"].astype(str).tolist()
batch_size = 64 if device.type == "cuda" else 8
print(f"✅ Batch size set to {batch_size}")

all_probs = []

# 6. 배치 예측
for i in tqdm(range(0, len(texts), batch_size), desc="감정 예측 진행중"):
    batch_texts = texts[i:i+batch_size]
    enc = tok(batch_texts, return_tensors="pt", padding=True,
              truncation=True, max_length=512).to(device)
    with torch.no_grad():
        logits = mdl(**enc).logits
        probs = torch.sigmoid(logits).cpu().numpy()
    all_probs.append(probs)

all_probs = np.vstack(all_probs)

# 7. 확률값 DataFrame으로 변환
probs_df = pd.DataFrame(all_probs, columns=[f"{lbl}_prob" for lbl in emotion_labels])

# 8. threshold=0.4 기준 라벨링도 추가
all_labels = []
for row in all_probs:
    indices = [i for i, p in enumerate(row) if p >= 0.4]
    labels = [emotion_labels[i] for i in indices]
    all_labels.append(labels)

df_sample["predicted_emotions"] = all_labels
df_sample["predicted_top1"] = [labels[0] if labels else None for labels in all_labels]

# 9. 원본 + 확률값 합치기
df_out = pd.concat([df_sample.reset_index(drop=True), probs_df.reset_index(drop=True)], axis=1)

# 10. 저장
df_out.to_csv("movie_long_with_emotions_sample300_probs.csv", index=False, encoding="utf-8-sig")
print("✅ 감정 예측 완료 → movie_long_with_emotions_sample300_probs.csv 저장")


C:\Users\82104\AppData\Local\Temp\ipykernel_10792\400644457.py:8: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("movie_long_micro_panel.csv")


✅ Running on cpu
✅ Batch size set to 8


감정 예측 진행중: 100%|██████████| 38/38 [00:20<00:00,  1.82it/s]

✅ 감정 예측 완료 → movie_long_with_emotions_sample300_probs.csv 저장


In [ ]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [13]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))

CUDA available: False
CUDA device count: 0


In [19]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# 1. 전체 데이터 로드
df = pd.read_csv("movie_long_micro_panel.csv")

# 2. 감상평이 비어있지 않은 행만 필터링
df_nonnull = df[df["감상평"].notna()].copy()
print(f"✅ 감상평이 있는 리뷰 수: {len(df_nonnull)}")

# 3. 모델 로드
model_id = "searle-j/kote_for_easygoing_people"
tok = AutoTokenizer.from_pretrained(model_id)
mdl = AutoModelForSequenceClassification.from_pretrained(model_id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mdl.to(device)
mdl.eval()
print(f"✅ Running on {device}")

# 4. 감정 라벨 목록
emotion_labels = [
 '불평/불만','환영/호의','감동/감탄','지긋지긋','고마움','슬픔','화남/분노','존경','기대감','우쭐댐/무시함',
 '안타까움/실망','비장함','의심/불신','뿌듯함','편안/쾌적','신기함/관심','아껴주는','부끄러움','공포/무서움','절망',
 '한심함','역겨움/징그러움','짜증','어이없음','없음','패배/자기혐오','귀찮음','힘듦/지침','즐거움/신남','깨달음',
 '죄책감','증오/혐오','흐뭇함(귀여움/예쁨)','당황/난처','경악','부담/안_내킴','서러움','재미없음','불쌍함/연민','놀람',
 '행복','불안/걱정','기쁨','안심/신뢰'
]

# 5. 텍스트 리스트
texts = df_nonnull["감상평"].astype(str).tolist()

# 👉 GPU에서는 batch_size=64~128 추천
batch_size = 64 if device.type == "cuda" else 8
print(f"✅ Batch size set to {batch_size}")
print(f"✅ 총 {len(texts)}개의 리뷰 예측 시작")

all_probs = []

# 6. 배치 예측
for i in tqdm(range(0, len(texts), batch_size), desc="감정 예측 진행중"):
    batch_texts = texts[i:i+batch_size]
    enc = tok(batch_texts, return_tensors="pt", padding=True,
              truncation=True, max_length=512).to(device)
    with torch.no_grad():
        logits = mdl(**enc).logits
        probs = torch.sigmoid(logits).cpu().numpy()
    all_probs.append(probs)

all_probs = np.vstack(all_probs)

# 7. 확률값 DataFrame으로 변환
probs_df = pd.DataFrame(all_probs, columns=[f"{lbl}_prob" for lbl in emotion_labels])

# 8. threshold=0.4 기준 라벨링
all_labels = []
for row in all_probs:
    indices = [i for i, p in enumerate(row) if p >= 0.4]
    labels = [emotion_labels[i] for i in indices]
    all_labels.append(labels)

df_nonnull["predicted_emotions"] = all_labels
df_nonnull["predicted_top1"] = [labels[0] if labels else None for labels in all_labels]

# 9. 원본 + 확률값 합치기
df_out = pd.concat([df_nonnull.reset_index(drop=True), probs_df.reset_index(drop=True)], axis=1)

# 10. 저장
df_out.to_csv("movie_long_with_emotions_full.csv", index=False, encoding="utf-8-sig")
print("✅ 전체 데이터 감정 예측 완료 → movie_long_with_emotions_full.csv 저장")

C:\Users\82104\AppData\Local\Temp\ipykernel_21328\2614438494.py:8: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("movie_long_micro_panel.csv")


✅ 감상평이 있는 리뷰 수: 74634
✅ Running on cuda
✅ Batch size set to 64
✅ 총 74634개의 리뷰 예측 시작


감정 예측 진행중: 100%|██████████| 1167/1167 [20:10<00:00,  1.04s/it] 


✅ 전체 데이터 감정 예측 완료 → movie_long_with_emotions_full.csv 저장


# KCELC

In [18]:
import transformers, torch
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)


transformers: 4.56.1
torch: 2.6.0+cu124


In [20]:
import transformers, inspect
print("transformers file:", transformers.__file__)
from transformers import TrainingArguments
print("TA file:", TrainingArguments.__module__)
print(inspect.signature(TrainingArguments.__init__))  # 인자 목록 확인


transformers file: C:\Users\82104\anaconda3\envs\hanyang_paper\Lib\site-packages\transformers\__init__.py
TA file: transformers.training_args
(self, output_dir: Optional[str] = None, overwrite_output_dir: bool = False, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, per_gpu_train_batch_size: Optional[int] = None, per_gpu_eval_batch_size: Optional[int] = None, gradient_accumulation_steps: int = 1, eval_accumulation_steps: Optional[int] = None, eval_delay: Optional[float] = 0, torch_empty_cache_steps: Optional[int] = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: Union[transformers.tr

In [7]:
pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Note: you may need to restart the kernel to use updated packages.Looking in indexes: https://download.pytorch.org/whl/cu124
   ---------------------------------------- 0.0/2.5 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 GB 5.6 MB/s eta 0:07:34
   ---------------------------------------- 0.0/2.5 GB 6.6 MB/s eta 0:06:25
   ---------------------------------------- 0.0/2.5 GB 7.1 MB/s eta 0:05:58
   ---------------------------------------- 0.0/2.5 GB 7.6 MB/s eta 0:05:34
   ---------------------------------------- 0.0/2.5 GB 7.7 MB/s eta 0:05:27
   ---------------------------------------- 0.0/2.5 GB 7.8 MB/s eta 0:05:22
   ---------------------------------------- 0.0/2.5 GB 7.9 MB/s eta 0:05:19
   ---------------------------------------- 0.0/2.5 GB 7.9 MB/s eta 0:05:20
   ---------------------------------------- 0.0/2.5 GB 7.9 MB/s eta 0:05:20
   ---------------------------------------- 0.0/2.5 GB 7.9 MB/s eta 0:05:20
   ---------------------------------------- 0.0

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kobert 0.2.4 requires mxnet<=1.7.0.post2,>=1.4.0, which is not installed.
kobert 0.2.4 requires onnxruntime, which is not installed.
kobert 0.2.4 requires sentencepiece<=0.1.96,>=0.1.6, but you have sentencepiece 0.2.1 which is incompatible.


In [11]:
import torch; print(torch.__version__)  # 2.6.x 이상이면 OK

2.5.1+cu121


In [40]:
# =========================
# KcELECTRA × KOTE(44) Fine-tuning Full Script (No-Error)
# =========================
import os, random, numpy as np, pandas as pd, torch, inspect
from pathlib import Path
from tqdm.auto import tqdm

print(f"🔧 Device: {'CUDA ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("torch:", torch.__version__)

# ---- 경로 수정: KOTE 폴더 ----
BASE = Path(r"C:\Users\82104\anaconda_projects_files(2025)\hanyang_paper(RE)\KOTE")
train_fp = BASE / "train.tsv"
valid_fp = BASE / "val.tsv"   # dev.tsv면 이 줄만 바꾸세요
test_fp  = BASE / "test.tsv"

# ---- 하이퍼파라미터 ----
NUM_LABELS = 44
MAX_LEN    = 256
LR         = 3e-5
EPOCHS     = 5
BS_TRAIN   = 32
BS_EVAL    = 64
SEED       = 42
MODEL_ID   = "beomi/KcELECTRA-base-v2022"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# =========================
# 1) KOTE TSV Loader (헤더 유무 자동)
# =========================
from datasets import Dataset, DatasetDict

def _parse_labels(x):
    # "0,3,41" / "0 3 41" / "[0,3,41]" 등 처리
    if pd.isna(x): return []
    s = str(x).strip()
    if not s: return []
    s = s.strip("[]()")
    s = s.replace(";", ",").replace(" ", ",")
    return [int(t) for t in s.split(",") if t.strip().isdigit()]

def read_tsv(fp: Path) -> Dataset:
    # 1) 헤더 있다고 가정
    df = pd.read_csv(fp, sep="\t", encoding="utf-8", engine="python")
    has_text_like = any(c in df.columns for c in ["text","document","sentence","content"])
    has_labels_like = any(c in df.columns for c in ["labels","label","targets","emotion_ids"])
    # 2) 없으면 헤더 없이 재읽기
    if not (has_text_like and has_labels_like):
        df = pd.read_csv(fp, sep="\t", encoding="utf-8", engine="python",
                         header=None, names=["id","text","labels"], on_bad_lines="skip")

    # 컬럼 정규화
    cols = list(df.columns)
    text_col = "text" if "text" in cols else (
        "document" if "document" in cols else
        "sentence" if "sentence" in cols else
        "content"  if "content"  in cols else "text"
    )
    if text_col != "text":
        df.rename(columns={text_col:"text"}, inplace=True)

    if "labels" not in df.columns:
        for cand in ["label","targets","emotion_ids"]:
            if cand in df.columns:
                df.rename(columns={cand:"labels"}, inplace=True)
                break

    if df["labels"].dtype == object:
        df["labels"] = df["labels"].apply(_parse_labels)

    if "id" not in df.columns:
        df["id"] = np.arange(len(df))

    return Dataset.from_pandas(df[["id","text","labels"]], preserve_index=False)

ds = DatasetDict(
    train=read_tsv(train_fp),
    validation=read_tsv(valid_fp),
    test=read_tsv(test_fp)
)
print(ds)

# =========================
# 2) 멀티핫 변환 + 토크나이즈
# =========================
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

def to_multihot(batch):
    # float32로 바로 생성 (BCEWithLogitsLoss 요구사항)
    arr = np.zeros((len(batch["labels"]), NUM_LABELS), dtype="float32")
    for i, labs in enumerate(batch["labels"]):
        for j in labs:
            if 0 <= j < NUM_LABELS:
                arr[i, j] = 1.0
    batch["labels"] = arr
    return batch

ds = ds.map(to_multihot, batched=True)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
def tokenize(b):
    return tok(b["text"], truncation=True, max_length=MAX_LEN)

ds = ds.map(tokenize, batched=True, remove_columns=["text"])

# === 라벨 float32 보장 + torch 텐서 포맷 ===
def cast_labels_to_float(example):
    return {"labels": np.asarray(example["labels"], dtype="float32")}
ds = ds.map(cast_labels_to_float)

cols = ["input_ids", "attention_mask", "labels"]
if "token_type_ids" in ds["train"].features:
    cols.insert(2, "token_type_ids")
ds.set_format(type="torch", columns=cols)

collator = DataCollatorWithPadding(tokenizer=tok)

# =========================
# 3) 모델 로드(멀티라벨 44)
# =========================
mdl = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)
# 분류 헤드 초기화 경고는 정상입니다(학습 진행하면 사라짐).

# =========================
# 4) 지표 함수
# =========================
from sklearn.metrics import f1_score

def compute_metrics(eval_pred, thresholds=(0.4,)):
    logits, labels = eval_pred
    probs = 1/(1+np.exp(-logits))
    out = {}
    for t in thresholds:
        pred = (probs >= t).astype(int)
        out[f"macro_f1@{t}"]   = f1_score(labels, pred, average="macro",  zero_division=0)
        out[f"micro_f1@{t}"]   = f1_score(labels, pred, average="micro",  zero_division=0)
        out[f"samples_f1@{t}"] = f1_score(labels, pred, average="samples",zero_division=0)
    return out

# =========================
# 5) TrainingArguments (eval/evaluation 호환)
# =========================
from transformers import TrainingArguments

ta_sig = inspect.signature(TrainingArguments.__init__)
kwargs = dict(
    output_dir="kc_electra_kote44_out",
    learning_rate=LR,
    per_device_train_batch_size=BS_TRAIN,
    per_device_eval_batch_size=BS_EVAL,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1@0.4",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    bf16=torch.cuda.is_available(),
    report_to="none",
    disable_tqdm=False,
)
if "evaluation_strategy" in ta_sig.parameters:
    kwargs["evaluation_strategy"] = "epoch"
else:
    kwargs["eval_strategy"] = "epoch"

args = TrainingArguments(**kwargs)


🔧 Device: CUDA NVIDIA GeForce RTX 3050 Ti Laptop GPU
torch: 2.6.0+cu124
DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'labels'],
        num_rows: 40000
    })
    validation: Dataset({
        features: ['id', 'text', 'labels'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'text', 'labels'],
        num_rows: 5000
    })
})


Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base-v2022 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [60]:
# ===== 6) Trainer & Train (커스텀 collate_fn로 NumPy 2.x 이슈 회피) =====
from transformers import Trainer, EarlyStoppingCallback, DataCollatorWithPadding
import torch, numpy as np

# 기본 패딩 콜레이터
base_collator = DataCollatorWithPadding(tokenizer=tok)

def collate_fn(features):
    """
    - inputs: base_collator로 패딩/텐서화
    - labels: float32 텐서로 수동 스택 (멀티라벨 BCEWithLogitsLoss 요구사항)
    - NumPy 2.x에서 datasets→numpy 변환 이슈 회피
    """
    # labels를 먼저 따로 모아 float32 텐서로
    labels = torch.tensor(
        [np.asarray(f["labels"], dtype=np.float32) for f in features],
        dtype=torch.float32
    )
    # labels 제외하고 패딩
    no_labels = [{k: v for k, v in f.items() if k != "labels"} for f in features]
    batch = base_collator(no_labels)
    batch["labels"] = labels
    return batch

trainer = Trainer(
    model=mdl,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    processing_class=tok,                     # tokenizer 경고 제거
    data_collator=collate_fn,                 # ✅ 커스텀 콜레이터
    compute_metrics=lambda p: compute_metrics(p, thresholds=(0.3, 0.4, 0.5)),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

train_out = trainer.train()
print(train_out)

C:\Users\82104\AppData\Local\Temp\ipykernel_22056\802601853.py:15: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  labels = torch.tensor(


Epoch,Training Loss,Validation Loss


τ search (VALID):   0%|          | 0/9 [00:00<?, ?it/s]

✅ VALID best τ=0.45 | Macro-F1=0.2901


In [58]:
# =========================
# 8) 테스트 성능 평가 (best τ + 참고값들)
# =========================
test_pred   = trainer.predict(ds["test"])
test_logits = test_pred.predictions
test_probs  = 1.0 / (1.0 + np.exp(-test_logits))
test_labels = np.asarray(ds["test"]["labels"], dtype=np.float32)

for t in [best_thr, 0.3, 0.4, 0.5]:
    t = float(t)
    pred = (test_probs >= t).astype(int)
    macro   = f1_score(test_labels, pred, average="macro",  zero_division=0)
    micro   = f1_score(test_labels, pred, average="micro",  zero_division=0)
    samples = f1_score(test_labels, pred, average="samples",zero_division=0)
    print(f"[TEST] τ={t:.2f} | Macro-F1={macro:.4f} | Micro-F1={micro:.4f} | Samples-F1={samples:.4f}")


[TEST] τ=0.45 | Macro-F1=0.2922 | Micro-F1=0.3054 | Samples-F1=0.2995
[TEST] τ=0.30 | Macro-F1=0.2921 | Micro-F1=0.3054 | Samples-F1=0.2995
[TEST] τ=0.40 | Macro-F1=0.2921 | Micro-F1=0.3054 | Samples-F1=0.2995
[TEST] τ=0.50 | Macro-F1=0.2062 | Micro-F1=0.2556 | Samples-F1=0.2467


In [ ]:
# ============================================
# KcELECTRA × KOTE(44) 멀티라벨 파인튜닝 전체 스크립트 (안전/강화판)
# ============================================
import os, random, numpy as np, pandas as pd, torch, inspect
from pathlib import Path
from tqdm.auto import tqdm

print(f"🔧 Device: {'CUDA ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("torch:", torch.__version__)

# ---------- 경로/하이퍼 ----------
BASE = Path(r"C:\Users\82104\anaconda_projects_files(2025)\hanyang_paper(RE)\KOTE")  # ← 본인 경로로 수정
train_fp = BASE / "train.tsv"
valid_fp = BASE / "val.tsv"     # dev.tsv면 여기 수정
test_fp  = BASE / "test.tsv"

NUM_LABELS = 44
MAX_LEN    = 512                # 긴 리뷰 보존
LR         = 2e-5
EPOCHS     = 5
ACC_STEPS  = 2
BS_TRAIN   = 16                 # 유효 배치 32 (=16×2)
BS_EVAL    = 64
SEED       = 42
MODEL_ID   = "beomi/KcELECTRA-base-v2022"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# =========================
# 1) KOTE TSV Loader (헤더 유무 자동)
# =========================
from datasets import Dataset, DatasetDict

def _parse_labels(x):
    # "0,3,41" / "0 3 41" / "[0,3,41]" 등 처리
    if pd.isna(x): return []
    s = str(x).strip()
    if not s: return []
    s = s.strip("[]()")
    s = s.replace(";", ",").replace(" ", ",")
    return [int(t) for t in s.split(",") if t.strip().isdigit()]

def read_tsv(fp: Path) -> Dataset:
    # 1) 헤더 가정
    df = pd.read_csv(fp, sep="\t", encoding="utf-8", engine="python")
    has_text_like = any(c in df.columns for c in ["text","document","sentence","content"])
    has_labels_like = any(c in df.columns for c in ["labels","label","targets","emotion_ids"])
    # 2) 없으면 헤더 없이 재읽기
    if not (has_text_like and has_labels_like):
        df = pd.read_csv(
            fp, sep="\t", encoding="utf-8", engine="python",
            header=None, names=["id","text","labels"], on_bad_lines="skip"
        )

    # 컬럼 정규화
    cols = list(df.columns)
    text_col = "text" if "text" in cols else (
        "document" if "document" in cols else
        "sentence" if "sentence" in cols else
        "content"  if "content"  in cols else "text"
    )
    if text_col != "text":
        df.rename(columns={text_col: "text"}, inplace=True)

    if "labels" not in df.columns:
        for cand in ["label","targets","emotion_ids"]:
            if cand in df.columns:
                df.rename(columns={cand: "labels"}, inplace=True)  # ← 오타 수정 완료
                break

    if df["labels"].dtype == object:
        df["labels"] = df["labels"].apply(_parse_labels)

    if "id" not in df.columns:
        df["id"] = np.arange(len(df))

    return Dataset.from_pandas(df[["id","text","labels"]], preserve_index=False)

ds = DatasetDict(
    train=read_tsv(train_fp),
    validation=read_tsv(valid_fp),
    test=read_tsv(test_fp)
)
print(ds)

# =========================
# 2) 멀티핫 변환 + 토크나이즈
# =========================
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def to_multihot(batch):
    arr = np.zeros((len(batch["labels"]), NUM_LABELS), dtype="float32")
    for i, labs in enumerate(batch["labels"]):
        for j in labs:
            if 0 <= j < NUM_LABELS:
                arr[i, j] = 1.0
    batch["labels"] = arr
    return batch

ds = ds.map(to_multihot, batched=True, load_from_cache_file=False)

tok = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize(b):
    return tok(b["text"], truncation=True, max_length=MAX_LEN)

ds = ds.map(tokenize, batched=True, remove_columns=["text"], load_from_cache_file=False)

# =========================
# 3) 모델 로드(멀티라벨 44)
# =========================
mdl = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)  # 분류 헤드 초기화 경고는 정상

# =========================
# 4) Metrics
# =========================
from sklearn.metrics import f1_score

def compute_metrics(eval_pred, thresholds=(0.4,)):
    logits, labels = eval_pred
    probs = 1/(1+np.exp(-logits))
    out = {}
    for t in thresholds:
        pred = (probs >= t).astype(int)
        out[f"macro_f1@{t}"]   = f1_score(labels, pred, average="macro",  zero_division=0)
        out[f"micro_f1@{t}"]   = f1_score(labels, pred, average="micro",  zero_division=0)
        out[f"samples_f1@{t}"] = f1_score(labels, pred, average="samples",zero_division=0)
    return out

# =========================
# 5) TrainingArguments (eval/evaluation 호환)
# =========================
from transformers import TrainingArguments
ta_sig = inspect.signature(TrainingArguments.__init__)
kwargs = dict(
    output_dir="kc_electra_kote44_out",
    learning_rate=LR,
    per_device_train_batch_size=BS_TRAIN,
    per_device_eval_batch_size=BS_EVAL,
    num_train_epochs=EPOCHS,
    gradient_accumulation_steps=ACC_STEPS,
    weight_decay=0.01,
    warmup_ratio=0.10,
    lr_scheduler_type="cosine",
    save_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1@0.4",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    bf16=torch.cuda.is_available(),
    report_to="none",
    disable_tqdm=False,
)
if "evaluation_strategy" in ta_sig.parameters:
    kwargs["evaluation_strategy"] = "steps"
else:
    kwargs["eval_strategy"] = "steps"

args = TrainingArguments(**kwargs)

# =========================
# 6) Trainer & Train — NumPy 2.x/라벨/불균형 안전판
# =========================
from transformers import Trainer, EarlyStoppingCallback, DataCollatorWithPadding
from datasets import Sequence, Value

# (A) 포맷 리셋 + python 포맷
try:
    ds.reset_format()
except Exception:
    pass
ds = ds.with_format("python")

# (B) 라벨을 파이썬 리스트(float)로 강제
def ensure_pylist_float_labels(batch):
    out = []
    for row in batch["labels"]:
        if hasattr(row, "tolist"):
            row = row.tolist()
        out.append([float(v) for v in row])
    return {"labels": out}

ds = ds.map(ensure_pylist_float_labels, batched=True, load_from_cache_file=False)

# (C) features에 labels=float32 시퀀스 명시(선택이지만 권장)
new_features = ds["train"].features.copy()
new_features["labels"] = Sequence(Value("float32"))
ds = ds.cast(new_features)

# (D) pos_weight 계산 (train 기준) — 클래스 불균형 보정
train_labels = np.asarray(ds["train"]["labels"], dtype=np.float32)  # (N, 44)
N = train_labels.shape[0]
pos = train_labels.sum(axis=0)
neg = N - pos
pos_safe = np.where(pos > 0, pos, 1.0)
pos_weight_vec = (neg / pos_safe).astype(np.float32)

# (E) collate_fn: 입력은 패딩 콜레이터, 라벨은 float32 텐서
base_collator = DataCollatorWithPadding(tokenizer=tok)

def collate_fn(features):
    labels = torch.tensor([f["labels"] for f in features], dtype=torch.float32)  # (B, 44)
    no_labels = [{k: v for k, v in f.items() if k != "labels"} for f in features]
    batch = base_collator(no_labels)
    batch["labels"] = labels
    return batch

# (F) pos_weight 지원 Trainer (register_buffer 미사용 버전)
class BCEWithPosWeightTrainer(Trainer):
    def __init__(self, *args, pos_weight: np.ndarray = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32) if pos_weight is not None else None

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")         # (B, 44) float32
        outputs = model(**inputs)
        logits = outputs.logits               # (B, 44)
        if self.pos_weight_tensor is not None:
            loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight_tensor.to(logits.device))
        else:
            loss_fct = torch.nn.BCEWithLogitsLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = BCEWithPosWeightTrainer(
    model=mdl,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    processing_class=tok,                           # tokenizer deprecated 경고 제거
    data_collator=collate_fn,                       # NumPy 2.x 안전
    compute_metrics=lambda p: compute_metrics(p, thresholds=(0.3, 0.4, 0.5)),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    pos_weight=pos_weight_vec,
)

train_out = trainer.train()
print(train_out)

# =========================
# 7) 검증 임계값 τ 탐색 (Macro-F1 기준)
# =========================
from sklearn.metrics import f1_score
val_pred = trainer.predict(ds["validation"])
val_logits = val_pred.predictions
val_probs  = 1.0 / (1.0 + np.exp(-val_logits))
val_labels = np.asarray(ds["validation"]["labels"], dtype=np.float32)

candidates = np.linspace(0.2, 0.6, 9)
best_thr, best_macro = None, -1.0
for t in tqdm(candidates, desc="τ search (VALID)"):
    pred = (val_probs >= float(t)).astype(int)
    macro = f1_score(val_labels, pred, average="macro", zero_division=0)
    if macro > best_macro:
        best_macro, best_thr = macro, float(t)
print(f"✅ VALID best τ={best_thr:.2f} | Macro-F1={best_macro:.4f}")

# =========================
# 8) 테스트 평가
# =========================
test_pred   = trainer.predict(ds["test"])
test_logits = test_pred.predictions
test_probs  = 1.0 / (1.0 + np.exp(-test_logits))
test_labels = np.asarray(ds["test"]["labels"], dtype=np.float32)

for t in [best_thr, 0.3, 0.4, 0.5]:
    t = float(t)
    pred = (test_probs >= t).astype(int)
    macro   = f1_score(test_labels, pred, average="macro",  zero_division=0)
    micro   = f1_score(test_labels, pred, average="micro",  zero_division=0)
    samples = f1_score(test_labels, pred, average="samples",zero_division=0)
    print(f"[TEST] τ={t:.2f} | Macro-F1={macro:.4f} | Micro-F1={micro:.4f} | Samples-F1={samples:.4f}")

# =========================
# 9) 모델 저장
# =========================
SAVE_DIR = "kc_electra_kote44_finetuned"
trainer.model.save_pretrained(SAVE_DIR)
tok.save_pretrained(SAVE_DIR)
print("💾 Saved to", SAVE_DIR)


🔧 Device: CUDA NVIDIA GeForce RTX 3050 Ti Laptop GPU
torch: 2.6.0+cu124
DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'labels'],
        num_rows: 40000
    })
    validation: Dataset({
        features: ['id', 'text', 'labels'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'text', 'labels'],
        num_rows: 5000
    })
})


Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/40000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]